In [1]:
import duckdb

con = duckdb.connect()

# 1. Comparativa colectiva del partido
df_partido = con.execute("""
    SELECT 
        team_name,
        team_side,
        SUM(goals_total) AS goles,
        ROUND(SUM(xg_total), 2) AS xg_generado,
        ROUND(SUM(xgot_total), 2) AS xgot_tiros_puerta,
        SUM(shots_total) AS tiros,
        ROUND(SUM(xt_total), 2) AS amenaza_xt,
        ROUND(SUM(xt_passes), 2) AS xt_pases,
        ROUND(SUM(xt_carries), 2) AS xt_conducciones,
        SUM(passes_completed) AS pases_acertados,
        SUM(progressive_passes) AS pases_progresivos,
        SUM(recoveries) AS recuperaciones,
        SUM(interceptions) AS intercepciones,
        SUM(tackles_won) AS entradas_ganadas
    FROM '../data/gold/player_match_gold.parquet'
    WHERE round = 3 AND (team_name ILIKE '%Hasselt%' OR team_name ILIKE '%Gent%')
    GROUP BY team_name, team_side
""").df()

print(df_partido.to_string())

                      team_name team_side  goles  xg_generado  xgot_tiros_puerta  tiros  amenaza_xt  xt_pases  xt_conducciones  pases_acertados  pases_progresivos  recuperaciones  intercepciones  entradas_ganadas
0          KAA Gent Reserve U23      away    2.0         1.48               2.08    8.0       -0.25     -1.18             0.64            348.0              133.0            54.0            13.0               8.0
1  Koninklijke Sporting Hasselt      home    0.0         2.36               0.64   20.0       -1.19     -2.00             0.38            439.0              183.0            65.0             8.0              12.0


In [2]:
# 2. Rendimiento individual en el partido
df_individual = con.execute("""
    SELECT 
        team_name,
        player_name,
        position,
        minutes_played,
        ROUND(xt_total, 3) AS xt,
        progressive_passes AS prog_passes,
        key_passes,
        shots_total AS tiros,
        ROUND(xg_total, 2) AS xg,
        goals_total AS goles,
        recoveries AS recuperaciones,
        tackles_won AS entradas
    FROM '../data/gold/player_match_gold.parquet'
    WHERE round = 3 AND (team_name ILIKE '%Hasselt%' OR team_name ILIKE '%Gent%')
      AND minutes_played > 45
    ORDER BY xt_total DESC
""").df()

print(df_individual.head(10).to_string())

                      team_name        player_name position  minutes_played     xt  prog_passes  key_passes  tiros    xg  goles  recuperaciones  entradas
0          KAA Gent Reserve U23      Ruslan Vydysh        M              74  0.281            7           2      2  0.96      1               3         0
1  Koninklijke Sporting Hasselt       Cheick Thiam        D              90  0.128           27           1      0  0.00      0               4         2
2          KAA Gent Reserve U23    Gilles De Meyer        D              89  0.091            7           0      1  0.03      0               5         0
3          KAA Gent Reserve U23  Gyano Vanderdonck        M              90  0.059            7           0      0  0.00      0               5         2
4          KAA Gent Reserve U23          Bas Evers        G              90  0.045           27           0      0  0.00      0              10         0
5  Koninklijke Sporting Hasselt        Livio Milts        M              76 

In [6]:
import duckdb
from pathlib import Path
import pandas as pd

# Ajustar según si el notebook se ejecuta desde notebooks/ o desde la raíz
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

SILVER_DIR = BASE_DIR / "data" / "silver"
GOLD_DIR = BASE_DIR / "data" / "gold"

con = duckdb.connect()

TEAM = "Koninklijke Sporting Hasselt"
print(f"[OK] Conexión abierta. Auditando datos para: {TEAM}")

query_integrity = f"""
WITH silver_counts AS (
    SELECT 
        COUNT(DISTINCT match_id) AS matches_silver,
        COUNT(DISTINCT player_id) AS players_silver,
        SUM(minutes_played) AS total_minutes_silver
    FROM read_parquet('{SILVER_DIR}/lineups_silver.parquet')
    WHERE team_name = '{TEAM}'
),
gold_counts AS (
    SELECT 
        COUNT(DISTINCT match_id) AS matches_gold,
        COUNT(DISTINCT player_id) AS players_gold,
        SUM(minutes_played) AS total_minutes_gold
    FROM read_parquet('{GOLD_DIR}/player_match_gold.parquet')
    WHERE team_name = '{TEAM}'
)
SELECT 
    s.*, 
    g.*,
    (s.total_minutes_silver = g.total_minutes_gold) AS minutes_match,
    (s.matches_silver = g.matches_gold) AS matches_match
FROM silver_counts s, gold_counts g;
"""

df_integrity = con.execute(query_integrity).df()
display(df_integrity)

# Control de nulos en métricas nucleares
query_nulls = f"""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE xt_total IS NULL) AS null_xt,
    COUNT(*) FILTER (WHERE xg_total IS NULL) AS null_xg,
    COUNT(*) FILTER (WHERE passes_completed > passes_total) AS impossible_passes
FROM read_parquet('{GOLD_DIR}/player_match_gold.parquet')
WHERE team_name = '{TEAM}';
"""
display(con.execute(query_nulls).df())

query_team = f"""
SELECT 
    round AS jor,
    opponent_name,
    team_side AS side,
    match_result AS res,
    goals_for || ' - ' || goals_against AS score,
    ROUND(xg_for, 2) AS xg_f,
    ROUND(xg_against, 2) AS xg_c,
    ROUND(xg_diff, 2) AS diff_xg,
    ROUND(xt_total, 2) AS xt_gen,
    ROUND(xt_conceded, 2) AS xt_con,
    ROUND(xt_turnover_risk, 2) AS risk_lost,
    possession_proxy_pct AS poss_pct,
    shots_for || ' (' || goals_for || ')' AS shots_goals
FROM read_parquet('{GOLD_DIR}/team_match_gold.parquet')
WHERE team_name = '{TEAM}'
ORDER BY round;
"""

df_team = con.execute(query_team).df()
display(df_team)

query_grain = f"""
SELECT 
    player_id,
    player_name,
    position,
    total_minutes,
    matches_played,
    matches_started,
    sub_appearances,
    passes_p90,
    pass_accuracy_pct,
    prog_passes_p90,
    xt_p90,
    xt_turnover_risk_p90,
    risk_reward_ratio,
    shots_p90,
    xg_p90,
    goals_total
FROM read_parquet('{GOLD_DIR}/master_player_gold.parquet')
WHERE team_name = '{TEAM}'
ORDER BY player_name, position DESC;
"""

df_master = con.execute(query_grain).df()
display(df_master.head(15))

# Chequeo específico: ¿Algún jugador sumó minutos en múltiples posiciones?
query_multipos = f"""
SELECT 
    player_name,
    COUNT(*) AS total_records, -- Si es > 2, jugó en 2+ demarcaciones además de TOTAL
    STRING_AGG(position, ', ') AS positions_logged
FROM read_parquet('{GOLD_DIR}/master_player_gold.parquet')
WHERE team_name = '{TEAM}'
GROUP BY player_name
HAVING COUNT(*) > 2;
"""
print("Jugadores polivalentes (multidemarcación):")
display(con.execute(query_multipos).df())

query_scouting = f"""
SELECT 
    player_name,
    position,
    total_minutes,
    -- Creación y progresión
    pct_xt,
    pct_prog_passes,
    pct_key_passes,
    pct_dribbles,
    -- Toma de decisiones y pase
    pct_pass_acc,
    pct_decision_making,
    -- Fase defensiva
    pct_recoveries,
    pct_tackles_won,
    -- Finalización
    pct_xg,
    pct_sav_quality
FROM read_parquet('{GOLD_DIR}/scouting_percentiles_gold.parquet')
WHERE team_name = '{TEAM}'
ORDER BY total_minutes DESC;
"""

df_scouting = con.execute(query_scouting).df()
display(df_scouting)

[OK] Conexión abierta. Auditando datos para: Koninklijke Sporting Hasselt


,matches_silver,players_silver,total_minutes_silver,matches_gold,players_gold,total_minutes_gold,minutes_match,matches_match
0,6,25,5998.0,6,25,5998.0,True,True


,total_rows,null_xt,null_xg,impossible_passes
0,120,0,0,0


,jor,opponent_name,side,res,score,xg_f,xg_c,diff_xg,xt_gen,xt_con,risk_lost,poss_pct,shots_goals
0,1,FCV Dender,home,W,1.0 - 0.0,1.93,1.94,-0.01,1.40,3.08,2.36,38.5,13.0 (1.0)
1,2,Royal Francs Borains,away,W,1.0 - 0.0,1.18,0.93,0.25,1.76,2.08,2.74,60.7,11.0 (1.0)
2,3,KAA Gent Reserve U23,home,L,0.0 - 2.0,2.36,1.48,0.88,3.03,1.74,3.83,55.8,20.0 (0.0)
3,4,Jong KRC Genk U23,away,L,0.0 - 3.0,2.26,2.05,0.21,3.53,1.87,4.08,65.0,21.0 (0.0)
4,5,RFC Seraing,away,W,2.0 - 1.0,1.73,1.03,0.70,2.30,1.68,2.65,60.9,13.0 (2.0)
5,6,K. Beerschot V.A.,home,L,0.0 - 1.0,0.73,3.06,-2.33,2.39,1.67,2.40,56.2,14.0 (0.0)


,player_id,player_name,position,total_minutes,matches_played,matches_started,sub_appearances,passes_p90,pass_accuracy_pct,prog_passes_p90,xt_p90,xt_turnover_risk_p90,risk_reward_ratio,shots_p90,xg_p90,goals_total
0,1101427,Alain-Jerry Matoka,TOTAL,92.0,6,0,5,74.35,67.1,25.43,0.3973,0.9097,0.44,1.96,0.5543,0.0
1,1101427,Alain-Jerry Matoka,D,92.0,6,0,5,74.35,67.1,25.43,0.3973,0.9097,0.44,1.96,0.5543,0.0
2,1413409,Andres Labie,TOTAL,540.0,6,6,0,68.17,93.2,23.00,0.1371,0.0692,1.98,0.17,0.0060,0.0
3,1413409,Andres Labie,D,540.0,6,6,0,68.17,93.2,23.00,0.1371,0.0692,1.98,0.17,0.0060,0.0
4,1457538,Cheick Thiam,TOTAL,540.0,6,6,0,62.00,90.3,24.50,0.1908,0.1282,1.49,0.33,0.0135,0.0
5,1457538,Cheick Thiam,D,540.0,6,6,0,62.00,90.3,24.50,0.1908,0.1282,1.49,0.33,0.0135,0.0
6,1396101,Ethan Butera,TOTAL,53.0,5,0,4,86.60,82.4,30.57,0.5004,0.5074,0.99,1.70,0.5638,0.0
7,1396101,Ethan Butera,D,53.0,5,0,4,86.60,82.4,30.57,0.5004,0.5074,0.99,1.70,0.5638,0.0
8,58917,Geoffry Hairemans,TOTAL,113.0,2,1,1,71.68,82.2,27.08,0.5550,0.3520,1.58,2.39,0.4352,0.0
9,58917,Geoffry Hairemans,M,29.0,1,0,1,90.00,79.3,40.34,0.5366,0.4444,1.21,6.21,1.5961,0.0


Jugadores polivalentes (multidemarcación):


,player_name,total_records,positions_logged
0,Javan Ngoyi,3,"F, M, TOTAL"
1,Tuur Dierckx,3,"F, M, TOTAL"
2,Mehdi Bounou,3,"F, M, TOTAL"
3,Geoffry Hairemans,3,"F, M, TOTAL"


,player_name,position,total_minutes,pct_xt,pct_prog_passes,pct_key_passes,pct_dribbles,pct_pass_acc,pct_decision_making,pct_recoveries,pct_tackles_won,pct_xg,pct_sav_quality
0,Cheick Thiam,D,540.0,61.5,84.6,21.5,24.6,83.1,78.5,36.9,47.7,29.2,44.4
1,Andres Labie,D,540.0,32.3,78.5,21.5,0.0,98.5,87.7,24.6,47.7,21.5,29.6
2,Lucas Vankerkhoven,M,523.0,80.3,98.6,28.2,19.7,93.0,81.7,57.7,16.9,31.0,64.2
3,Lucca Lucker,D,496.0,52.3,49.2,46.2,56.9,44.6,29.2,50.8,32.3,16.9,46.3
4,Nicolas Gerits,M,492.0,29.6,84.5,23.9,22.5,50.7,18.3,81.7,33.8,40.8,14.9
5,Loic Masscho,D,476.0,83.1,26.2,98.5,96.9,15.4,24.6,95.4,67.7,75.4,66.7
6,Livio Milts,M,422.0,84.5,63.4,49.3,56.3,2.8,15.5,36.6,93.0,69.0,35.8
7,Siebe Van Keymolen,F,394.0,27.6,58.6,0.0,79.3,27.6,10.3,37.9,58.6,65.5,27.6
8,Mehdi Bounou,F,359.0,82.8,82.8,96.6,31.0,24.1,58.6,86.2,69.0,6.9,72.4
9,Javan Ngoyi,M,264.0,87.3,8.5,74.6,90.1,21.1,56.3,73.2,2.8,91.5,31.3
